# Actividad — Predicción de precios de viviendas con modelos lineales y polinomiales
**Aprendizaje Supervisado · Sesión 4 · Regresión lineal y polinomial**

**Qué haremos.** En la presentación vimos cómo **mínimos cuadrados (OLS)** elige la recta que hace pequeño el error, cómo el **descenso de gradiente**
llega al mismo resultado dando pasos, cómo los **polinomios** doblan la recta y cómo el **grado** decide entre subajuste y sobreajuste.
Hoy lo aplicas todo a los **precios de vivienda de California** (el mismo conjunto de la Sesión 1): ¿cuánto mejora una recta a «predecir siempre el promedio»?
¿llega tu descenso de gradiente al mismo lugar que `LinearRegression`? ¿qué grado de polinomio conviene y cómo se ven sus errores?

| Parte | Pregunta que responde | Tema de la presentación |
|---|---|---|
| 0 | ¿Qué datos tenemos y cómo los dividimos? | Nuestro caso |
| 1 | ¿Cuál es la línea base que todo modelo debe superar? | El objetivo |
| 2 | ¿Qué dicen los coeficientes de una recta simple y de una múltiple? | A1–A7 |
| 3 | ¿Llega el descenso de gradiente a la solución de OLS? ¿Qué hace la tasa $\eta$? | B1–B3 |
| 4 | ¿Qué pasa con `SGDRegressor` si no escalamos? | B5–B7 |
| 5 | ¿Qué grado de polinomio elegir con validación cruzada? | C1–C7 |
| 6 | ¿Cómo rinde el modelo elegido en prueba y qué dicen sus residuos? | D1–D4, E1 |
| 7 | Conclusión final | Resumen |
| 8 | Reto (bonificación): ¿y si no quitamos los extremos? | C6 |

**Tiempo estimado:** 60–75 minutos de trabajo individual.

**Entregable:** este notebook ejecutado (con tus respuestas en las celdas ✏️) y el archivo `entregable_unidad4.csv`
con la tabla *modelo · columnas · RMSE (validación cruzada) · MAE · RMSE · $R^2$ en prueba · comentario* de la Parte 6.

**Cómo trabajar.** Ejecuta las celdas en orden. Donde dice ✏️ hay una línea de código por completar o una pregunta;
escribe tus respuestas en la celda de texto que sigue, bajo «**Respuesta:**».

> ⚠️ **Regla de oro:** el conjunto de **prueba** (`Xte`, `yte`) se usa **una sola vez**, en la Parte 6. Todas las comparaciones anteriores
> (línea base, recta, polinomios) se hacen con **validación cruzada** sobre el entrenamiento.

> 💡 `fetch_california_housing` descarga el archivo (≈ 1 MB) la primera vez y lo guarda en `~/scikit_learn_data`; se necesita conexión a internet.

## Parte 0 — Los datos

Cada fila es un **distrito** censal de California (censo de 1990). El objetivo `MedHouseVal` es el valor mediano de las viviendas en **cientos de miles de dólares** (2.0 = 200 000 USD) y tiene un **tope en 5.0**. Usaremos 7 variables: `MedInc` (ingreso mediano, en decenas de miles de USD), `HouseAge`, `AveRooms`, `AveOccup`, `Population`, `Latitude` y `Longitude`.

In [ ]:
import warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 140)

VARIABLES = ["MedInc", "HouseAge", "AveRooms", "AveOccup", "Population", "Latitude", "Longitude"]
datos = fetch_california_housing(as_frame=True).frame
print("Distritos:", datos.shape[0])
datos[VARIABLES + ["MedHouseVal"]].describe().T.round(2)

Como vimos en el EDA de la Sesión 1, hay distritos con valores **extremos** (hasta 1243 ocupantes por hogar). Los quitamos con las mismas reglas de la presentación y después separamos el **20 % para prueba**.

In [ ]:
extremo = (datos["AveRooms"] >= 10) | (datos["AveOccup"] >= 6) | (datos["Population"] >= 10000)
limpio = ...     # ✏️ conserva solo las filas que NO son extremas: datos[~extremo]
print(f"Quitamos {extremo.sum()} distritos extremos; quedan {len(limpio)}")

X, y = limpio[VARIABLES], limpio["MedHouseVal"]
Xtr, Xte, ytr, yte = ...     # ✏️ train_test_split(X, y, test_size=0.2, random_state=42)
print("Entrenamiento:", Xtr.shape, "  Prueba:", Xte.shape)
print(f"Distritos en el tope de 5.0 (entrenamiento): {(ytr >= 5).sum()}   valor medio: {ytr.mean():.3f}")

# Protocolo de validación que usaremos SIEMPRE (el conjunto de prueba no se toca hasta la Parte 6)
kf = KFold(n_splits=5, shuffle=True, random_state=0)
ESCALAS = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}


def evalua_cv(modelo, Xd=None):
    # Promedio de la validación cruzada de 5 particiones sobre el ENTRENAMIENTO: MAE, RMSE y R2
    Xd = Xtr if Xd is None else Xd
    r = cross_validate(modelo, Xd, ytr, cv=kf, scoring=ESCALAS)
    return {"MAE": -r["test_MAE"].mean(), "RMSE": -r["test_RMSE"].mean(), "R2": r["test_R2"].mean()}


tabla_cv = {}    # aquí iremos guardando el resultado de cada modelo

## Parte 1 — La línea base  *(presentación: «El objetivo»)*

El modelo más tonto posible: predecir **siempre el promedio** del entrenamiento, sin mirar ninguna variable. `DummyRegressor` hace justo eso. Cualquier modelo que no lo supere no sirve.

In [ ]:
base = ...     # ✏️ DummyRegressor(strategy="mean")
tabla_cv["Siempre el promedio"] = evalua_cv(base)
pd.DataFrame(tabla_cv).T.round(3)

✏️ **Pregunta 1.1.** (a) Traduce el MAE y el RMSE de la línea base a dólares. (b) ¿Por qué su $R^2$ es prácticamente 0? (c) ¿Por qué el RMSE es mayor que el MAE?

**Respuesta:**

## Parte 2 — Regresión lineal: simple y múltiple  *(presentación A1–A7)*

Primero una recta con una sola variable, el ingreso: $\widehat{\texttt{MedHouseVal}} = b + w\cdot\texttt{MedInc}$.

In [ ]:
simple = ...     # ✏️ LinearRegression().fit(Xtr[["MedInc"]], ytr)
print(f"Recta aprendida:  valor = {simple.intercept_:.3f} + {simple.coef_[0]:.3f} · MedInc")
tabla_cv["Recta con MedInc"] = evalua_cv(LinearRegression(), Xtr[["MedInc"]])

fig, ax = plt.subplots(figsize=(6.2, 3.4))
muestra = Xtr.sample(2000, random_state=0).index
ax.scatter(Xtr.loc[muestra, "MedInc"], ytr.loc[muestra], s=5, alpha=0.3, color="gray")
xs = np.linspace(0, 15, 50)
ax.plot(xs, simple.intercept_ + simple.coef_[0] * xs, color="C3", lw=2)
ax.set_xlabel("MedInc (× 10 000 USD)"); ax.set_ylabel("MedHouseVal (× 100 000 USD)"); plt.tight_layout(); plt.show()
pd.DataFrame(tabla_cv).T.round(3)

✏️ **Pregunta 2.1.** (a) Interpreta la pendiente en dólares. (b) ¿Qué valor predice la recta para un distrito con `MedInc` = 5? Escríbelo en USD. (c) ¿Cuánto bajó el RMSE de validación respecto a la línea base? (d) Mirando la gráfica, ¿qué patrón de los datos no puede captar una recta?

**Respuesta:**

Ahora con las **7 variables**. Ajustamos dos veces el mismo modelo: con las variables en sus unidades originales y con las variables **estandarizadas** (media 0, desviación 1), que es como se pueden **comparar** los pesos entre sí.

In [ ]:
multiple = ...     # ✏️ LinearRegression().fit(Xtr, ytr)
estandar = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)

coefs = pd.DataFrame({"unidades originales": multiple.coef_,
                      "estandarizado": estandar[-1].coef_,
                      "correlación con y": [np.corrcoef(Xtr[v], ytr)[0, 1] for v in VARIABLES]},
                     index=VARIABLES).round(4)
print("Intercepto (unidades originales):", round(multiple.intercept_, 3))
print(coefs.to_string())
tabla_cv["Lineal con 7 variables"] = evalua_cv(LinearRegression())

fig, ax = plt.subplots(figsize=(6.2, 3.0))
orden = coefs["estandarizado"].abs().sort_values().index
ax.barh(orden, coefs.loc[orden, "estandarizado"], color=["C0" if v > 0 else "C3" for v in coefs.loc[orden, "estandarizado"]])
ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("coeficiente con variables estandarizadas"); plt.tight_layout(); plt.show()
pd.DataFrame(tabla_cv).T.round(3)

✏️ **Pregunta 2.2.** (a) Con las variables estandarizadas, ¿cuáles son las tres que más pesan y con qué signo? ¿Qué significa el signo negativo de `Latitude` y `Longitude`? (b) ¿Por qué el coeficiente de `Population` en unidades originales es casi 0? ¿Quiere decir que no importa? (c) Compara el signo del coeficiente de `AveRooms` con el de su correlación con $y$. ¿Por qué pueden ser distintos? (Pista: «manteniendo fijas las demás»). (d) ¿Cuánto mejoró el RMSE de validación al pasar de 1 a 7 variables?

**Respuesta:**

## Parte 3 — Descenso de gradiente por lotes con NumPy  *(presentación A3, B1–B3)*

Vamos a programar nosotros mismos el algoritmo. Usamos las 7 variables **estandarizadas** y agregamos una columna de unos para el intercepto, de modo que $\hat{\mathbf y}=Z\boldsymbol\theta$ con $\boldsymbol\theta=(b,w_1,\dots,w_7)$. Primero la **solución cerrada** (ecuación normal) $\hat{\boldsymbol\theta}=(Z^\top Z)^{-1}Z^\top\mathbf y$, que será nuestra referencia. (Con `np.linalg.solve` se resuelve el sistema sin calcular la inversa.)

In [ ]:
escalador = StandardScaler().fit(Xtr)
Z = np.c_[np.ones(len(Xtr)), escalador.transform(Xtr)]     # primera columna de unos = intercepto
yv = ytr.to_numpy()
NOMBRES = ["intercepto"] + VARIABLES

theta_ols = ...     # ✏️ np.linalg.solve(Z.T @ Z, Z.T @ yv)
mse_ols = np.mean((Z @ theta_ols - yv) ** 2)
comparacion = pd.DataFrame({"ecuación normal": theta_ols,
                            "LinearRegression": np.r_[estandar[-1].intercept_, estandar[-1].coef_]}, index=NOMBRES)
print(comparacion.round(4).to_string())
print(f"MSE de entrenamiento de OLS = {mse_ols:.4f}")

Ahora el **descenso de gradiente por lotes**: en cada iteración se usan **todos** los datos para calcular el gradiente del error cuadrático medio

$$\nabla J(\boldsymbol\theta)=\frac{2}{n}\,Z^\top\big(Z\boldsymbol\theta-\mathbf y\big),\qquad \boldsymbol\theta\leftarrow\boldsymbol\theta-\eta\,\nabla J(\boldsymbol\theta).$$

Completa la línea del gradiente y prueba **tres tasas** de aprendizaje: $\eta=0.01$, $0.3$ y $0.5$, partiendo siempre de $\boldsymbol\theta=\mathbf 0$.

In [ ]:
def descenso_lotes(Z, y, eta, n_iter=300):
    theta = np.zeros(Z.shape[1])
    historia = []                                   # MSE de entrenamiento después de cada paso
    for t in range(n_iter):
        residuo = Z @ theta - y
        gradiente = ...     # ✏️ (2 / len(y)) * Z.T @ residuo
        theta = theta - eta * gradiente
        historia.append(np.mean((Z @ theta - y) ** 2))
    return theta, np.array(historia)


TASAS = [0.01, 0.3, 0.5]
historias, filas = {}, []
for eta in TASAS:
    theta, h = descenso_lotes(Z, yv, eta)
    historias[eta] = h
    filas.append({"eta": eta, "MSE tras 10": h[9], "MSE tras 50": h[49], "MSE tras 300": h[-1],
                  "máx |theta - OLS|": np.abs(theta - theta_ols).max()})
print(pd.DataFrame(filas).to_string(index=False, float_format=lambda v: f"{v:.4g}"))

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for eta in TASAS:
    ax.plot(np.arange(1, 301), np.minimum(historias[eta], 1e3), label=f"η = {eta}")
ax.axhline(mse_ols, color="k", ls=":", lw=1, label="OLS")
ax.set_yscale("log"); ax.set_xlabel("iteración"); ax.set_ylabel("MSE de entrenamiento (cortado en 1000)"); ax.legend(); plt.tight_layout(); plt.show()

✏️ **Pregunta 3.1.** (a) Describe qué pasa con cada tasa: ¿cuál es lenta, cuál adecuada y cuál diverge? (b) Con la tasa adecuada, ¿llega el descenso de gradiente a los mismos coeficientes que la ecuación normal? ¿Cómo lo sabes? (c) Con $\eta=0.01$, ¿cuántas iteraciones crees que harían falta para llegar a OLS? Compruébalo cambiando `n_iter`.

**Respuesta:**

¿Dónde está la frontera entre converger y diverger? Esta celda prueba tasas entre 0.40 y 0.52.

In [ ]:
for eta in np.round(np.arange(0.40, 0.53, 0.02), 2):
    _, h = descenso_lotes(Z, yv, eta, n_iter=500)
    print(f"eta = {eta:.2f}   MSE tras 500 iteraciones = {h[-1]:.4g}")

# Dato teórico: el descenso converge si eta < 2 / (mayor valor propio de (2/n) Z^T Z)
valores_propios = np.linalg.eigvalsh(2 / len(yv) * Z.T @ Z)
print(f"\nUmbral teórico: eta < {2 / valores_propios.max():.3f}")

✏️ **Pregunta 3.2.** (a) ¿Entre qué dos tasas pasa el algoritmo de converger a diverger? ¿Coincide con el umbral teórico? (b) Si `LinearRegression` resuelve el problema de una vez, ¿por qué molestarse con el descenso de gradiente? Da dos razones de la presentación.

**Respuesta:**

## Parte 4 — `SGDRegressor` con y sin escalado  *(presentación B5–B7)*

`SGDRegressor` es el descenso de gradiente **estocástico** de scikit-learn: actualiza los pesos con un dato a la vez y usa una tasa que **decrece** con el tiempo (`invscaling`). Lo comparamos con y sin `StandardScaler` en el pipeline, con la misma configuración de la presentación.

In [ ]:
config = dict(loss="squared_error", learning_rate="invscaling", eta0=0.01, max_iter=1000, tol=1e-4, random_state=0)

sgd_con = ...     # ✏️ make_pipeline(StandardScaler(), SGDRegressor(**config))
sgd_sin = SGDRegressor(**config)

tabla_cv["SGDRegressor con StandardScaler"] = evalua_cv(sgd_con)
tabla_cv["SGDRegressor sin escalar"] = evalua_cv(sgd_sin)
print(pd.DataFrame(tabla_cv).T.loc[["Lineal con 7 variables", "SGDRegressor con StandardScaler", "SGDRegressor sin escalar"]]
      .to_string(float_format=lambda v: f"{v:.4g}"))

sgd_con.fit(Xtr, ytr); sgd_sin.fit(Xtr, ytr)
print("\nCoeficientes con escalado (estandarizados):", np.round(sgd_con[-1].coef_, 3))
print("Coeficientes de OLS (estandarizados):       ", np.round(estandar[-1].coef_, 3))
print("Coeficientes sin escalar:                   ", np.array2string(sgd_sin.coef_, precision=2, max_line_width=200))
print("\nRango de cada variable:", (Xtr.max() - Xtr.min()).round(1).to_dict())

✏️ **Pregunta 4.1.** (a) Con escalado, ¿qué tan parecido es `SGDRegressor` a `LinearRegression` (métricas y coeficientes)? (b) ¿Qué le pasa sin escalar? Explica por qué usando el rango de `Population` frente al de `MedInc`. (c) ¿Por qué el `StandardScaler` debe ir **dentro** del pipeline y no aplicarse antes a todo `X`?

**Respuesta:**

## Parte 5 — Regresión polinomial: ¿qué grado?  *(presentación C1–C7)*

Con un `Pipeline` **escalar → crear potencias e interacciones → OLS** probamos grados de 1 a 5. Con 7 variables y grado $d$ hay $\binom{7+d}{d}-1$ columnas. Guardamos el RMSE de **entrenamiento** y de **validación** de cada partición para ver la brecha entre ambos.

In [ ]:
filas, por_particion = [], {}
for grado in range(1, 6):
    modelo = ...     # ✏️ make_pipeline(StandardScaler(), PolynomialFeatures(grado, include_bias=False), LinearRegression())
    r = cross_validate(modelo, Xtr, ytr, cv=kf, scoring="neg_root_mean_squared_error", return_train_score=True)
    por_particion[grado] = -r["test_score"]
    filas.append({"grado": grado, "columnas": math.comb(7 + grado, grado) - 1,
                  "rmse_entrenamiento": -r["train_score"].mean(), "rmse_validacion": -r["test_score"].mean(),
                  "desv_validacion": r["test_score"].std()})
grados = pd.DataFrame(filas).set_index("grado")
print(grados.round(3).to_string())
print("\nRMSE de validación en cada partición:")
print(pd.DataFrame(por_particion, index=[f"partición {i + 1}" for i in range(5)]).round(3).to_string())

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(grados.index, grados["rmse_entrenamiento"], "o--", label="entrenamiento")
ax.errorbar(grados.index, grados["rmse_validacion"], yerr=grados["desv_validacion"], fmt="o-", capsize=3, label="validación cruzada")
ax.set_xticks(grados.index); ax.set_xlabel("grado del polinomio"); ax.set_ylabel("RMSE"); ax.legend(); plt.tight_layout(); plt.show()

✏️ **Pregunta 5.1.** (a) ¿Qué pasa con el error de entrenamiento al subir el grado? ¿Y con el de validación? (b) ¿En qué grado empieza el sobreajuste y cómo lo reconoces en la tabla (brecha entrenamiento–validación)? (c) Mira el RMSE por partición: ¿qué grado es el menos **estable** y qué te dice eso sobre su varianza?

**Respuesta:**

Elige el grado con **validación cruzada** (no con la prueba). Si dos grados están muy cerca, recuerda que el más simple es preferible.

In [ ]:
GRADO = ...     # ✏️ escribe el grado que elegiste (un entero entre 1 y 5)
print(f"Grado elegido: {GRADO}  ({math.comb(7 + GRADO, GRADO) - 1} columnas)   RMSE de validación = {grados.loc[GRADO, 'rmse_validacion']:.3f}")

✏️ **Pregunta 5.2.** Justifica tu elección de grado en 2 o 3 líneas: ¿qué ganas frente al grado anterior y qué arriesgas frente al siguiente?

**Respuesta:**

## Parte 6 — Evaluación final en prueba y análisis de residuos  *(presentación D1–D4, E1)*

Ahora sí, **una sola vez**: entrenamos cada modelo con **todo** el entrenamiento y lo evaluamos en el conjunto de prueba. Incluimos también el RMSE de validación cruzada para comparar lo que *esperábamos* con lo que *obtuvimos*.

In [ ]:
def metricas(y_real, y_pred):
    return {"MAE": mean_absolute_error(y_real, y_pred), "RMSE": mean_squared_error(y_real, y_pred) ** 0.5, "R2": r2_score(y_real, y_pred)}


finales = {
    "Siempre el promedio": (DummyRegressor(strategy="mean"), ["MedInc"], 0),
    "Recta con MedInc": (LinearRegression(), ["MedInc"], 1),
    "Lineal con 7 variables": (LinearRegression(), VARIABLES, 7),
    f"Polinomial grado {GRADO}": (make_pipeline(StandardScaler(), PolynomialFeatures(GRADO, include_bias=False), LinearRegression()),
                                  VARIABLES, math.comb(7 + GRADO, GRADO) - 1),
}
filas = []
for nombre, (modelo, cols, ncol) in finales.items():
    modelo.fit(Xtr[cols], ytr)
    cv_rmse = tabla_cv[nombre]["RMSE"] if nombre in tabla_cv else grados.loc[GRADO, "rmse_validacion"]
    m = metricas(yte, modelo.predict(Xte[cols]))     # <- único uso del conjunto de prueba
    filas.append({"modelo": nombre, "columnas": ncol, "rmse_cv": cv_rmse,
                  "mae_prueba": m["MAE"], "rmse_prueba": m["RMSE"], "r2_prueba": m["R2"]})
informe = pd.DataFrame(filas)
print(informe.round(3).to_string(index=False))

✏️ **Pregunta 6.1.** (a) ¿Se parece el RMSE de prueba al de validación cruzada en cada modelo? ¿Qué indicaría una diferencia grande? (b) ¿Hay sobreajuste en el modelo elegido? Justifica con las cifras de entrenamiento, validación y prueba. (c) Traduce el MAE y el RMSE del modelo elegido a dólares y compáralos con la línea base.

**Respuesta:**

Los residuos $e_i=y_i-\hat y_i$ cuentan lo que las métricas no dicen. Graficamos tres vistas del modelo elegido en el conjunto de prueba.

In [ ]:
elegido = finales[f"Polinomial grado {GRADO}"][0]
pred = elegido.predict(Xte)
residuo = yte - pred
tope = yte >= 5

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].scatter(yte, pred, s=4, alpha=0.3); ax[0].plot([0, 5.2], [0, 5.2], "k--", lw=1)
ax[0].set_xlabel("valor real"); ax[0].set_ylabel("predicción"); ax[0].set_title("predicción vs. real", fontsize=9)
ax[1].scatter(pred, residuo, s=4, alpha=0.3, c=np.where(tope, "C3", "C0")); ax[1].axhline(0, color="k", lw=1)
ax[1].set_xlabel("predicción"); ax[1].set_ylabel("residuo"); ax[1].set_title("residuos (rojo: distritos en el tope de 5.0)", fontsize=9)
ax[2].hist(residuo, bins=60, color="C0"); ax[2].set_xlabel("residuo"); ax[2].set_title("distribución de residuos", fontsize=9)
plt.tight_layout(); plt.show()

print(f"Residuo promedio: {residuo.mean():.3f}")
print(f"Distritos en el tope: {tope.sum()}   residuo promedio = {residuo[tope].mean():.3f}   resto: {residuo[~tope].mean():.3f}")
print(f"Residuos con |e| > 1.5 (más de 150 000 USD): {(residuo.abs() > 1.5).sum()} de {len(residuo)}")
print(f"RMSE sin los distritos del tope: {mean_squared_error(yte[~tope], pred[~tope]) ** 0.5:.3f}")

✏️ **Pregunta 6.2.** (a) ¿Los residuos están centrados en 0 y sin patrón? (b) ¿Qué es la banda diagonal de puntos rojos y por qué el modelo se equivoca sistemáticamente ahí? (c) ¿Cuánto cambia el RMSE si quitamos los distritos del tope? ¿Qué te dice eso sobre dónde está el error?

**Respuesta:**

✏️ **Pregunta 6.3.** Si tuvieras que reportar **una sola** métrica a una inmobiliaria, ¿cuál elegirías (MAE, RMSE o $R^2$) y por qué? ¿En qué caso preferirías la otra?

**Respuesta:**

In [ ]:
comentarios = {
    "Siempre el promedio": "✏️ escribe tu comentario",
    "Recta con MedInc": "✏️ escribe tu comentario",
    "Lineal con 7 variables": "✏️ escribe tu comentario",
    f"Polinomial grado {GRADO}": "✏️ escribe tu comentario",
}
entregable = informe.round(4).copy()
entregable["comentario"] = entregable["modelo"].map(comentarios)
entregable.to_csv("entregable_unidad4.csv", index=False, encoding="utf-8")
entregable

## Parte 7 — Conclusión final

✏️ **Pregunta 7.1.** En **6 a 8 líneas** resume para un colega: (1) qué modelo recomiendas y con qué pasos (limpieza, división, pipeline, elección del grado); (2) qué error esperar en prueba, con cifras y en dólares, frente a la línea base; (3) si hay sobreajuste o subajuste y cómo lo sabes; (4) qué limitación seguiría teniendo el modelo y qué intentarías después.

**Respuesta:**

## Parte 8 — Reto (5 puntos de bonificación): ¿y si no quitamos los extremos?

*Presentación C6.* Repetimos la validación cruzada del polinomio de grado 3 con **los 20 640 distritos originales** (misma división 80/20 y mismas particiones).

In [ ]:
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(datos[VARIABLES], datos["MedHouseVal"], test_size=0.2, random_state=42)
poli3 = make_pipeline(StandardScaler(), PolynomialFeatures(3, include_bias=False), LinearRegression())
r_sucio = ...     # ✏️ -cross_validate(poli3, Xtr_c, ytr_c, cv=kf, scoring="neg_root_mean_squared_error")["test_score"]
r_limpio = por_particion[3]
print("RMSE por partición SIN quitar extremos:", np.round(r_sucio, 2))
print("RMSE por partición CON datos limpios:  ", np.round(r_limpio, 3))
print(f"\nAveOccup máximo en los datos originales: {datos['AveOccup'].max():.0f}   y elevado al cubo: {datos['AveOccup'].max() ** 3:,.0f}")

✏️ **Pregunta del reto.** ¿Por qué algunas particiones dan un error enorme y otras no? ¿Qué hacen las potencias con un valor extremo, y por qué una recta (grado 1) sufriría mucho menos?

**Respuesta:**